# Grok Evaluation - With Reasoning Token Logging

This notebook evaluates Grok on the OI Benchmark and captures reasoning tokens.

**Install:** `pip install openai`

**Models:**
- `grok-3-mini` - Supports `reasoning_effort` (low/high)
- `grok-3` - No reasoning_effort parameter
- `grok-4` - Always max reasoning (no parameter needed)

In [ ]:
from openai import OpenAI

api_key = "YOUR_XAI_API_KEY"  # ← Your xAI API key

client = OpenAI(
    api_key=api_key,
    base_url="https://api.x.ai/v1"
)

In [ ]:
import pandas as pd
import os
import glob

# ============ BATCH PROCESSING OPTIONS ============
BATCH_MODE = True  # Set to True to process all files in a folder
RATA_FOLDER = "Benchmark/4-Intermediate_1/Translations_GPT5.2"  # Folder containing input files
RESULTS_FOLDER = "Results/translations_results_RATA/translations_results_grok"  # Where to save outputs
OUTPUT_PREFIX = "grok_4_1_fast_reasoning"  # Prefix for output files

# Single file mode settings (used when BATCH_MODE = False)
input_csv_path = "Benchmark/OP_Benchmark.csv"

if BATCH_MODE:
    # Find all rata_100_*.csv files
    all_files = glob.glob(os.path.join(RATA_FOLDER, "rata_100_*.csv"))
    all_files = sorted(all_files)
    
    # Ensure output folder exists
    os.makedirs(RESULTS_FOLDER, exist_ok=True)
    
    # Check which files have already been processed
    processed_files = []
    pending_files = []
    
    print(f"Found {len(all_files)} RATA translation files\n")
    print("Status check:")
    print("-" * 80)
    
    for file_path in all_files:
        filename = os.path.splitext(os.path.basename(file_path))[0]
        output_csv = os.path.join(RESULTS_FOLDER, f"{OUTPUT_PREFIX}_{filename}.csv")
        
        if os.path.exists(output_csv):
            try:
                output_df = pd.read_csv(output_csv)
                input_df = pd.read_csv(file_path)
                if len(output_df) == len(input_df) and output_df['answer_to_prompt_1'].notna().all() and output_df['answer_to_prompt_2'].notna().all():
                    processed_files.append(file_path)
                    print(f"{filename} - COMPLETE ({len(output_df)} rows)")
                else:
                    pending_files.append(file_path)
                    print(f"{filename} - INCOMPLETE ({output_df['answer_to_prompt_1'].notna().sum()}/{len(output_df)} rows)")
            except:
                pending_files.append(file_path)
                print(f"{filename} - ERROR reading output")
        else:
            pending_files.append(file_path)
            print(f"{filename} - NOT STARTED")
    
    print("-" * 80)
    print(f"\nSummary: {len(processed_files)} complete, {len(pending_files)} pending\n")
    
    if pending_files:
        print(f"Will process {len(pending_files)} files:")
        for f in pending_files:
            print(f"  - {os.path.basename(f)}")
    else:
        print("All files already processed!")
else:
    # Single file mode
    print(f"Single file mode: {os.path.basename(input_csv_path)}")
    pending_files = [input_csv_path]
    df = pd.read_csv(input_csv_path)
    print(f"Loaded {len(df)} questions")
    df.head()

In [ ]:
import time, threading
from collections import deque
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import APIError, RateLimitError, APITimeoutError

# ============ CONFIGURATION ============
model_name = "grok-4-1-fast-reasoning"  # Options: grok-3-mini, grok-3, grok-4, grok-4-1-fast-reasoning
REASONING_EFFORT = "high"  # Only for grok-3-mini: "low" or "high"
USE_REASONING_EFFORT = False  # Set to True only for grok-3-mini

MAX_WORKERS = 4
RPM_LIMIT = 30
SAVE_EVERY = 5

# ============ RATE LIMITER ============
class RPMLimiter:
    def __init__(self, rpm):
        self.rpm = rpm
        self.calls = deque()
        self.lock = threading.Lock()

    def wait(self):
        while True:
            with self.lock:
                now = time.time()
                while self.calls and now - self.calls[0] >= 60:
                    self.calls.popleft()
                if len(self.calls) < self.rpm:
                    self.calls.append(now)
                    return
            time.sleep(0.05)

rpm_limiter = RPMLimiter(RPM_LIMIT)
save_lock = threading.Lock()

# ============ API CALL ============
def ask_until_success(prompt):
    """Returns: (answer_text, input_tokens, output_tokens, reasoning_tokens)"""
    while True:
        try:
            rpm_limiter.wait()
            
            request_params = {
                "model": model_name,
                "max_tokens": 8192,
                "messages": [{"role": "user", "content": prompt}]
            }
            
            if USE_REASONING_EFFORT:
                request_params["reasoning_effort"] = REASONING_EFFORT
            
            response = client.chat.completions.create(**request_params)
            
            text = response.choices[0].message.content.strip() if response.choices else ""
            
            input_tokens = 0
            output_tokens = 0
            reasoning_tokens = 0
            
            if hasattr(response, 'usage') and response.usage:
                input_tokens = getattr(response.usage, 'prompt_tokens', 0) or 0
                output_tokens = getattr(response.usage, 'completion_tokens', 0) or 0
                
                if hasattr(response.usage, 'completion_tokens_details'):
                    details = response.usage.completion_tokens_details
                    if details:
                        reasoning_tokens = getattr(details, 'reasoning_tokens', 0) or 0
            
            if text:
                return text, input_tokens, output_tokens, reasoning_tokens
            
            print("Empty response — retrying in 4s…")
            time.sleep(4)

        except RateLimitError as e:
            print(f"Rate limit — retrying in 15s: {e}")
            time.sleep(15)
        except (APIError, APITimeoutError) as e:
            print(f"API issue — retrying in 4s: {e}")
            time.sleep(4)
        except Exception as e:
            print(f"Error — retrying in 4s: {e}")
            time.sleep(4)

# ============ PROCESS A SINGLE FILE ============
def process_file(file_path):
    """Process a single file and save results"""
    filename = os.path.splitext(os.path.basename(file_path))[0]
    
    if BATCH_MODE:
        csv_path = os.path.join(RESULTS_FOLDER, f"{OUTPUT_PREFIX}_{filename}.csv")
    else:
        csv_path = f"{model_name.replace('-', '_')}_{filename}.csv"
    
    print(f"\n{'='*60}")
    print(f"Processing: {filename}")
    print(f"Output: {csv_path}")
    print(f"{'='*60}")
    
    # Load input file
    file_df = pd.read_csv(file_path)
    
    # Check if output exists and load partial progress
    if csv_path != file_path and os.path.exists(csv_path):
        file_df = pd.read_csv(csv_path)
        if 'answer_to_prompt_1' in file_df.columns:
            print(f"Loaded existing progress: {file_df['answer_to_prompt_1'].notna().sum()}/{len(file_df)} rows complete")
    
    # Add columns if missing
    for col in ['answer_to_prompt_1', 'answer_to_prompt_2', 
                'input_tokens_1', 'input_tokens_2',
                'output_tokens_1', 'output_tokens_2',
                'reasoning_tokens_1', 'reasoning_tokens_2']:
        if col not in file_df.columns:
            file_df[col] = None
    
    file_df.reset_index(drop=True, inplace=True)
    
    def save_file():
        with save_lock:
            file_df.to_csv(csv_path, index=False)
    
    row_lock = threading.Lock()
    progress = {"count": 0}
    
    def process_row(i):
        if pd.notna(file_df.at[i, 'answer_to_prompt_1']) and pd.notna(file_df.at[i, 'answer_to_prompt_2']):
            return

        p1 = str(file_df.at[i, 'prompt.1'])
        p2 = str(file_df.at[i, 'prompt.2'])
        opt = str(file_df.at[i, 'OPTIONS'])

        if "OPTIONS" in p1: p1 = p1.replace("OPTIONS", opt)
        if "OPTIONS" in p2: p2 = p2.replace("OPTIONS", opt)

        a1, in1, out1, reason1 = ask_until_success(p1)
        a2, in2, out2, reason2 = ask_until_success(p2)

        with row_lock:
            file_df.at[i, 'answer_to_prompt_1'] = a1
            file_df.at[i, 'answer_to_prompt_2'] = a2
            file_df.at[i, 'input_tokens_1'] = in1
            file_df.at[i, 'input_tokens_2'] = in2
            file_df.at[i, 'output_tokens_1'] = out1
            file_df.at[i, 'output_tokens_2'] = out2
            file_df.at[i, 'reasoning_tokens_1'] = reason1
            file_df.at[i, 'reasoning_tokens_2'] = reason2
            progress["count"] += 1

            if progress["count"] % SAVE_EVERY == 0:
                save_file()
                print(f"[{filename}] Saved @ {progress['count']} rows | Reasoning: {reason1}, {reason2}")
    
    # Find rows to process
    todo = [i for i in range(len(file_df))
            if pd.isna(file_df.at[i, 'answer_to_prompt_1']) or pd.isna(file_df.at[i, 'answer_to_prompt_2'])]
    
    if not todo:
        print(f"[{filename}] Already complete!")
        return
    
    print(f"[{filename}] {len(todo)} rows to process")
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(process_row, i) for i in todo]
        for idx, _ in enumerate(as_completed(futures), 1):
            if idx % 10 == 0:
                print(f"[{filename}] Progress: {idx}/{len(todo)}")
    
    save_file()
    
    # Final verification
    missing = file_df[file_df['answer_to_prompt_1'].isna() | file_df['answer_to_prompt_2'].isna()].index.tolist()
    while missing:
        print(f"[{filename}] Fixing {len(missing)} missing...")
        for i in missing:
            process_row(i)
            save_file()
        missing = file_df[file_df['answer_to_prompt_1'].isna() | file_df['answer_to_prompt_2'].isna()].index.tolist()
    
    save_file()
    print(f"[{filename}] COMPLETE — Saved to {csv_path}")

# ============ MAIN LOOP ============
if pending_files:
    print(f"\nStarting processing of {len(pending_files)} files...")
    print(f"Model: {model_name}")
    if USE_REASONING_EFFORT:
        print(f"Reasoning effort: {REASONING_EFFORT}")
    
    for file_idx, file_path in enumerate(pending_files, 1):
        print(f"\n{'#'*60}")
        print(f"# FILE {file_idx}/{len(pending_files)}")
        print(f"{'#'*60}")
        process_file(file_path)
    
    print(f"\n{'='*60}")
    print(f"ALL {len(pending_files)} FILES PROCESSED!")
    print(f"{'='*60}")
else:
    print("No pending files to process!")